# Part 1: Forward Mode Automatic Differentiation

Forward mode AD can simply be implemented by defining a class to represent [dual numbers](https://en.wikipedia.org/wiki/Dual_number) which hold the value and its derivative. The following skeleton defines a dual number and implements multiplication.

__Tasks:__

- Addition (`__add__`) is incomplete - can you finish it?
- Can you also implement division (`__truediv__`), subtraction (`__sub__`) and power (`__pow__`)?

In [11]:
import math

import math

class DualNumber:
    def __init__(self, value, dvalue):
        self.value = value
        self.dvalue = dvalue

    def __str__(self):
        return str(self.value) + " + " + str(self.dvalue) + "ε"

    def __mul__(self, other):
        return DualNumber(self.value * other.value,
            self.dvalue * other.value + other.dvalue * self.value)

    def __add__(self, other):
        return DualNumber(self.value + other.value, self.dvalue + other.dvalue)

    def __sub__(self, other):
        return DualNumber(self.value - other.value, self.dvalue - other.dvalue)

    def __truediv__(self, other):
        return DualNumber(self.value / other.value,
            (self.dvalue * other.value - self.value * other.dvalue) / (other.value ** 2))

    def __pow__(self, other):
        if not isinstance(other, DualNumber):
            other = DualNumber(other, 0)   # treat plain numbers as constants
        value = self.value ** other.value
        dvalue = other.value * self.value ** (other.value - 1) * self.dvalue
        if other.dvalue != 0:
            dvalue += value * math.log(self.value) * other.dvalue
        return DualNumber(value, dvalue)



In [12]:
# Tests

DualNumber(1,0) + DualNumber(1,0) / DualNumber(1,0) - DualNumber(1,0)**DualNumber(1,0)


## Implementing math functions

We also need to implement some core math functions. Here's the sine function for a dual number:

In [13]:
def sin(x):
    return DualNumber(math.sin(x.value), math.cos(x.value)*x.dvalue)

__Task:__ can you implement the _cosine_ (`cos`), _tangent_ (`tan`), and _exponential_ (`exp`) functions in the code block below?

In [14]:
# TODO: implement additional math functions on dual numbers

def cos(x):
    return DualNumber(math.cos(x.value), -math.sin(x.value) * x.dvalue)

def tan(x):
    return DualNumber(math.tan(x.value), x.dvalue / math.cos(x.value) ** 2)

def exp(x):
    return DualNumber(math.exp(x.value), math.exp(x.value) * x.dvalue)

In [15]:
# Tests
assert cos(DualNumber(0,0)).value == 1
assert tan(DualNumber(0,0)).value == 0
assert exp(DualNumber(0,0)).value == 1


## Time to try it out

We're now in a position to try our implementation.

__Task:__

- Try running the following code to compute the value of the function $z=x\cdot y+sin(x)$ given $x=0.5$ and $y=4.2$, together with the derivative $\partial z/\partial x$ at that point.

In [17]:
x = DualNumber(0.5, 1)
y = DualNumber(4.2, 0)

z = x * y + sin(x)
print(z)


2.579425538604203 + 5.077582561890373ε


__Task__: Differentiate the above function with respect to $x$ and write the symbolic derivatives in the following box. Verify the result computed above is correct by plugging-in the values into your symbolic gradient expression.

5.077582562


__Task:__ Now use the code block below to compute the derivative $\partial z/\partial y$ of the above expression (at the same point $x=0.5, y=4.2$ as above) and store the derivative in the variable `dzdy` (just the derivative, not the Dual Number). Verify by hand that the result is correct.

In [18]:
x = DualNumber(0.5, 0)
y = DualNumber(4.2, 1)

z = x * y + sin(x)
dzdy = z.dvalue


print('dz/dy:', dzdy)

dz/dy: 0.5


In [19]:
#Tests
assert dzdy
assert type(dzdy) == float


__Task:__ Finally, use the code block below to experiment and test the other math functions and methods you created.

In [20]:
def check(name, f, df, a):
    result = f(DualNumber(a, 1))
    h = 1e-6
    numeric = (f(DualNumber(a + h, 0)).value - f(DualNumber(a - h, 0)).value) / (2 * h)
    print(f"{name:<22} value={result.value:.6f}  dual={result.dvalue:.6f}  "
          f"analytic={df(a):.6f}  numeric={numeric:.6f}")

a = 0.7
c = DualNumber(3.0, 0)   # a constant

check("x + 3",       lambda x: x + c,      lambda x: 1,                     a)
check("x - 3",       lambda x: x - c,      lambda x: 1,                     a)
check("x * x",       lambda x: x * x,      lambda x: 2 * x,                 a)
check("3 / x",       lambda x: c / x,      lambda x: -3 / x**2,             a)
check("x ** 3",      lambda x: x ** 3,     lambda x: 3 * x**2,              a)
check("x ** x",      lambda x: x ** x,     lambda x: x**x * (math.log(x) + 1), a)
check("cos(x)",      cos,                  lambda x: -math.sin(x),          a)
check("tan(x)",      tan,                  lambda x: 1 / math.cos(x)**2,    a)
check("exp(x)",      exp,                  lambda x: math.exp(x),           a)
check("exp(cos(x))", lambda x: exp(cos(x)),
                     lambda x: -math.sin(x) * math.exp(math.cos(x)),        a)
check("tan(x)/x**2", lambda x: tan(x) / x ** 2,
                     lambda x: (x**2 / math.cos(x)**2 - 2 * x * math.tan(x)) / x**4, a)

x + 3                  value=3.700000  dual=1.000000  analytic=1.000000  numeric=1.000000
x - 3                  value=-2.300000  dual=1.000000  analytic=1.000000  numeric=1.000000
x * x                  value=0.490000  dual=1.400000  analytic=1.400000  numeric=1.400000
3 / x                  value=4.285714  dual=-6.122449  analytic=-6.122449  numeric=-6.122449
x ** 3                 value=0.343000  dual=1.470000  analytic=1.470000  numeric=1.470000
x ** x                 value=0.779056  dual=0.501186  analytic=0.501186  numeric=0.501186
cos(x)                 value=0.764842  dual=-0.644218  analytic=-0.644218  numeric=-0.644218
tan(x)                 value=0.842288  dual=1.709450  analytic=1.709450  numeric=1.709450
exp(x)                 value=2.013753  dual=2.013753  analytic=2.013753  numeric=2.013753
exp(cos(x))            value=2.148655  dual=-1.384202  analytic=-1.384202  numeric=-1.384202
tan(x)/x**2            value=1.718956  dual=-1.422630  analytic=-1.422630  numeric=-1.4226